In [1]:
import openai
import pandas as pd
import fastembed

from qdrant_client import QdrantClient
from qdrant_client import models
from qdrant_client.models import VectorParams, Distance, PointStruct, PayloadSchemaType, SparseVectorParams, Document, Prefetch, FusionQuery



/Users/vivekkaushik/Desktop/amazon-chat-agent/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
qdrant_client = QdrantClient(url="http://localhost:6333") 

### Qdrant Collection for Hybrid search

In [3]:
qdrant_client.create_collection(
    collection_name="Amazon-items-collection-01-hybrid-search",
    vectors_config={"text-embedding-3-small":VectorParams(size=1536, distance=Distance.COSINE)},
    sparse_vectors_config={"bm25": SparseVectorParams(modifier=models.Modifier.IDF)}
)

True

In [4]:
qdrant_client.create_payload_index(
    collection_name="Amazon-items-collection-01-hybrid-search", 
    field_name="parent_asin",
    field_schema=PayloadSchemaType.KEYWORD,
    )

UpdateResult(operation_id=2, status=<UpdateStatus.COMPLETED: 'completed'>)

In [5]:
def get_embedding(text,model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=[text],
        model=model,
    )
    return response.data[0].embedding

In [6]:
def get_embeddings_batch(text_list, model="text-embedding-3-small",batch_size=100):
    if len(text_list)<=batch_size:
        response = openai.embeddings.create(
            input=text_list,
            model=model,
        )
        embeddings = [item.embedding for item in response.data]
        return embeddings
    embeddings = []
    counter = 1
    for i in range(0, len(text_list), batch_size):
        batch = text_list[i:i+batch_size]
        response = openai.embeddings.create(
            input=batch,
            model=model,
        )
        batch_embeddings = [item.embedding for item in response.data]
        embeddings.extend(batch_embeddings)
        print(f"Processed batch {counter} with {len(batch)} texts.")
        counter += 1
    return embeddings 

### Process and embed amazon items data

In [7]:
df_items = pd.read_json("../../data/meta_Electronics_2022_2023_has_main_category_sample_1000.jsonl", lines=True)

In [8]:
df_items.head()

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author
0,Camera & Photo,GuFamily Indoor Security Camera 2K HD 360 Degr...,1.7,8,[【2K Clarity for Full Coverage Monitoring】: Th...,[2K HD Indoor Surveillance Camera Works with 2...,18.99,[{'thumb': 'https://m.media-amazon.com/images/...,[],generic,"[Electronics, Camera & Photo, Video Surveillan...",{'Package Dimensions': '6.18 x 3.54 x 3.43 inc...,B0CBNZ93NJ,NaN,NaN,NaN
1,Cell Phones & Accessories,Tinkers Silicone Airpod Pro Case with Airtag H...,4.2,9,[Airpod Pro Keyring Case with Airtag Holder Fo...,[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': '2 in 1 Silicone Protective Case fo...,Tinkers,"[Electronics, Headphones, Earbuds & Accessorie...",{'Package Dimensions': '3.86 x 3.62 x 1.3 inch...,B09Q5HMPLS,NaN,NaN,NaN
2,All Electronics,Cameron sino 2200mAh 3.7V Li-ion 361-00023-13 ...,3.0,1,[Replacement for models: Garmin Pro 550 handhe...,[Cameron Sino is a brand focusing on sell batt...,25.78,[{'thumb': 'https://m.media-amazon.com/images/...,[],Cameron Sino®,[],{'Product Dimensions': '2.61 x 0.73 x 0.8 inch...,B01LZ85O5N,NaN,NaN,NaN
3,Computers,LTPRPTS Replacement Laptop LCD Back Cover Top ...,5.0,5,[1. 100% Original Brand New for Lenovo Flex 5 ...,[New Replacement for Lenovo Flex 5 15 5 15iil0...,55.68,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Installation video ONLY for IdeaPa...,LTPRPTS,"[Electronics, Computers & Accessories, Compute...",{'Product Dimensions': '13.78 x 9.84 x 3.94 in...,B0B7X9CQTF,NaN,NaN,NaN
4,Camera & Photo,ETLIN Security Camera Indoor with Smoke Detect...,4.5,112,[Security Camera Smoke Detector: 2-in-1 design...,"[Home Camera Featuring:, ●HD 1080P Video ●HD 1...",45.99,[{'thumb': 'https://m.media-amazon.com/images/...,[],ETLIN,"[Electronics, Camera & Photo, Video Surveillan...","{'Package Dimensions': '3 x 1.1 x 1.1 inches',...",B0C6GCBXMT,NaN,NaN,NaN


In [9]:
len(df_items)

1000

In [10]:
def preprocess_description(row):
    return f"{row['title']} {' '.join(row['features'])}"

In [11]:
def extract_first_large_image(row):
    return row['images'][0].get('large',"")

In [12]:
df_items['description'] = df_items.apply(preprocess_description, axis=1)
df_items['image'] = df_items.apply(extract_first_large_image, axis=1)

In [13]:
data_to_embed = df_items[['description','image','rating_number','price','average_rating','parent_asin']].to_dict(orient='records')

In [14]:
data_to_embed

[{'description': 'GuFamily Indoor Security Camera 2K HD 360 Degree Night Vision Camera, WiFi 2.4GHZ & 5GHz Home Camera with Motion Detection & Tracking for Baby and Pet/Dog/Cat Camera with APP, Cloud & SD Card Storage 【2K Clarity for Full Coverage Monitoring】: This indoor security camera boasts a 2K resolution, combined with 360-degree horizontal rotation and 105-degree vertical tilt, allowing you to effectively monitor your home. The full coverage monitoring of this home camera is not only for space,but also for time, the night vision function provides you with 24-hour continuous monitoring without interruption, so you can know what happened at night. 【2.4GHz&5GHzDual-Band WiFi Connection】: This home camera works with 2.4GHz & 5GHz WiFi, and easy to install, even can not install on the wall, you can just put it on the shelf or any flat area. It can be installed on the ceiling if needed. So you neither need to worry about whether this camera can connect to your network, nor distressed 

In [15]:
text_to_embed = [item['description'] for item in data_to_embed]

In [16]:
text_to_embed

['GuFamily Indoor Security Camera 2K HD 360 Degree Night Vision Camera, WiFi 2.4GHZ & 5GHz Home Camera with Motion Detection & Tracking for Baby and Pet/Dog/Cat Camera with APP, Cloud & SD Card Storage 【2K Clarity for Full Coverage Monitoring】: This indoor security camera boasts a 2K resolution, combined with 360-degree horizontal rotation and 105-degree vertical tilt, allowing you to effectively monitor your home. The full coverage monitoring of this home camera is not only for space,but also for time, the night vision function provides you with 24-hour continuous monitoring without interruption, so you can know what happened at night. 【2.4GHz&5GHzDual-Band WiFi Connection】: This home camera works with 2.4GHz & 5GHz WiFi, and easy to install, even can not install on the wall, you can just put it on the shelf or any flat area. It can be installed on the ceiling if needed. So you neither need to worry about whether this camera can connect to your network, nor distressed about how to ins

In [17]:
embeddings = get_embeddings_batch(text_to_embed)

Processed batch 1 with 100 texts.
Processed batch 2 with 100 texts.
Processed batch 3 with 100 texts.
Processed batch 4 with 100 texts.
Processed batch 5 with 100 texts.
Processed batch 6 with 100 texts.
Processed batch 7 with 100 texts.
Processed batch 8 with 100 texts.
Processed batch 9 with 100 texts.
Processed batch 10 with 100 texts.


In [18]:
len(embeddings)

1000

In [19]:
pointstructs = []
i=1
for embedding,data in zip(embeddings,data_to_embed):
    point = PointStruct(
        id=i,
        vector={
            "text-embedding-3-small": embedding,
            "bm25":Document(
                text=data['description'],
                model="qdrant/bm25"
             )
        },
        payload=data
    )
    pointstructs.append(point)
    i+=1

In [20]:
# pointstructs[0].vector
print(pointstructs[0].vector["bm25"])


text='GuFamily Indoor Security Camera 2K HD 360 Degree Night Vision Camera, WiFi 2.4GHZ & 5GHz Home Camera with Motion Detection & Tracking for Baby and Pet/Dog/Cat Camera with APP, Cloud & SD Card Storage 【2K Clarity for Full Coverage Monitoring】: This indoor security camera boasts a 2K resolution, combined with 360-degree horizontal rotation and 105-degree vertical tilt, allowing you to effectively monitor your home. The full coverage monitoring of this home camera is not only for space,but also for time, the night vision function provides you with 24-hour continuous monitoring without interruption, so you can know what happened at night. 【2.4GHz&5GHzDual-Band WiFi Connection】: This home camera works with 2.4GHz & 5GHz WiFi, and easy to install, even can not install on the wall, you can just put it on the shelf or any flat area. It can be installed on the ceiling if needed. So you neither need to worry about whether this camera can connect to your network, nor distressed about how to

In [21]:
bad_points = [
    (idx, p.vector["bm25"].model)
    for idx, p in enumerate(pointstructs)
    if p.vector["bm25"].model != "qdrant/bm25"
]

print(f"Bad points: {len(bad_points)}")

if bad_points:
    print(bad_points[:10])

Bad points: 0


In [22]:
models = {p.vector["bm25"].model for p in pointstructs}
print(models)
print(fastembed.__version__)

{'qdrant/bm25'}
0.8.0


In [23]:
qdrant_client.upsert(
    collection_name="Amazon-items-collection-01-hybrid-search",
    points=pointstructs,
    wait=True
)

UpdateResult(operation_id=3, status=<UpdateStatus.COMPLETED: 'completed'>)

### Hybrid Retrieval

In [24]:
def retrieve_data(query, qdrant_client, top_k=5):
    query_embedding = get_embedding(query)
    search_result = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01-hybrid-search",
        prefetch=[
            Prefetch(
                query=query_embedding,
                using='text-embedding-3-small',
                limit=20
            ),
            Prefetch(
                query=Document(
                    text=query, 
                    model="qdrant/bm25"),
                using='bm25',
                limit=20

            )
        ],
        query= FusionQuery(fusion='rrf'),
        limit=top_k,
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []
    for point in search_result.points:
        retrieved_context_ids.append(point.payload["parent_asin"])
        retrieved_context.append(point.payload["description"])
        similarity_scores.append(point.score)
        retrieved_context_ratings.append(point.payload["average_rating"])
    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }

In [25]:
results = retrieve_data("Do you have any security cameras?", qdrant_client, top_k=20)

In [26]:
results

{'retrieved_context_ids': ['B0BYNW99P8',
  'B0B51B52QH',
  'B0B44JW89D',
  'B09X7D59WX',
  'B0BCWBK73G',
  'B0C6GCBXMT',
  'B0BJYSFLDW',
  'B09ZPS99B1',
  'B0BZD7VRK6',
  'B0BWLVNN9W',
  'B0BJZPQJ82',
  'B0B2NPGZXX',
  'B0BRYXM3YK',
  'B0B8FZ2MR4',
  'B09Y949L5W',
  'B0C4P45FXV',
  'B0B9N7SYGF',
  'B0C2PJQ2H8',
  'B0C36LJTGM',
  'B0BL3KFW9L'],
 'retrieved_context': ['[24/7 Recording] 1080P Wireless Home Security Camera System, 4 x 2MP Outdoor Bullet Cameras + 8CH NVR, CCTV Camera Set for Smart Motion Detection, Plug & Play, Remote Control, IP66 Waterproof ⚙️[WIRELESS SECURITY CAMERA SYSTEM] With a private wireless protocol, you do not need to connect the cameras to NVR with a network cable or Wi-Fi. Viewing live recording just need to connect the NVR to router and monitor, then connect the power cords to cameras and NVR. Besides, the cameras can keep recording with wireless cascading even when far away from the NVR. The cameras could keep 24/7 recording to protect your home, garden, ga